Imports

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import warnings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import pandas as pd
import matplotlib.pyplot as plt
import MotionClouds as mc
import math

In [2]:
# Fixed parameters/hyperparameters ----------------------
N_X, N_Y, N_frames = 256, 256, 1
pix_per_degree = 11.1704
N_thetas = 50
thetas = np.linspace(0, np.pi, N_thetas, endpoint=False)
sf_base = 0.7 / pix_per_degree

B_thetas_deg = np.linspace(0.01, 45, 20) 
B_sfs_cpd = [0.01, 0.175, 0.7, 2.8]

B_thetas_rad = [np.deg2rad(deg) for deg in B_thetas_deg]
B_sfs_cpp = [sf / pix_per_degree for sf in B_sfs_cpd]

epochs = 150 # Epochs per model per grid point
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('output', exist_ok=True)

embed_dim = 32

In [3]:
# Model definitions -----------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=4), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(4),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, embed_dim),
            nn.LayerNorm(embed_dim)
        )
    def forward(self, x):
        return self.net(x)

class JEPAPredictor(nn.Module):
    def __init__(self, embed_dim=64, context_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim + context_dim, 128),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, embed_dim)
        )
    def forward(self, context, mask_info):
        return self.net(torch.cat([context, mask_info], dim=-1))

class PixelDecoder(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.fc = nn.Linear(embed_dim, 128 * 4 * 4) 
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=4), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 2, stride=2), nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 2, stride=2), nn.BatchNorm2d(16), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=4)
        )
    def forward(self, x):
        x = self.fc(x)
        x = x.view(-1, 128, 4, 4) 
        return self.net(x)


In [ ]:

    
import torch.distributed as dist

class SIGRegLoss(nn.Module):
    def __init__(self, num_slices=256):
        super().__init__()
        self.num_slices = num_slices

    def forward(self, x, global_step=0):
        dev = dict(device=x.device)
        g = torch.Generator(**dev)
        g.manual_seed(global_step)
        proj_shape = (x.size(1), self.num_slices)
        A = torch.randn(proj_shape, generator=g, **dev)
        A /= A.norm(p=2, dim=0)
        
        t = torch.linspace(-5, 5, 17, **dev)
        

        exp_f = torch.exp(-0.5 * t**2)

        x_t = (x @ A).unsqueeze(2) * t
        ecf = (1j * x_t).exp().mean(0)
        
        
        # Weighted L2 distance
        err = (ecf - exp_f).abs().square().mul(exp_f)
        
        N = x.size(0) 
        
        T = torch.trapz(err, t, dim=1) * N
        
        return T.mean()

In [5]:
class ImageDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx]

In [6]:
# Visualization -----------------------------------
def save_reconstruction_plot(originals, masked_inputs, ae_recons, sae_recons, jepa_recons, b_theta, b_sf):
    orig = originals[:5].cpu().numpy().squeeze()
    masked = masked_inputs[:5].cpu().numpy().squeeze()
    ae_rec = ae_recons[:5].detach().cpu().numpy().squeeze()
    sae_rec = sae_recons[:5].detach().cpu().numpy().squeeze()
    jepa_rec = jepa_recons[:5].detach().cpu().numpy().squeeze()
    
    fig, axes = plt.subplots(5, 5, figsize=(15, 15))
    fig.suptitle(f"Model Reconstructions | B_theta={b_theta}°, B_sf={b_sf} cpd", fontsize=16)
    
    for i in range(5):
        axes[0, i].imshow(orig[i], cmap='gray')
        axes[0, i].axis('off')
        if i == 0: axes[0, i].set_title("Original")
        
        axes[1, i].imshow(masked[i], cmap='gray')
        axes[1, i].axis('off')
        if i == 0: axes[1, i].set_title("Masked Input")
        
        axes[2, i].imshow(ae_rec[i], cmap='gray')
        axes[2, i].axis('off')
        if i == 0: axes[2, i].set_title("Decoded Standard MAE")

        axes[3, i].imshow(sae_rec[i], cmap='gray')
        axes[3, i].axis('off')
        if i == 0: axes[3, i].set_title("Decoded Sparse MAE")

        axes[4, i].imshow(jepa_rec[i], cmap='gray')
        axes[4, i].axis('off')
        if i == 0: axes[4, i].set_title("Decoded LeJEPA")
        
    plt.tight_layout()
    plt.savefig(f"output/recon_Btheta_{b_theta:.3f}_Bsf_{b_sf:.3f}.png")
    plt.close()
quadrant_coords = torch.tensor([[0,0], [0,1], [1,0], [1,1]], dtype=torch.float32).to(device)

In [7]:
# Probe Evaluation -------------------------------------
def evaluate_model(encoder_model, X_data, y_data):
    encoder_model.eval()
    X_tensor = torch.tensor(X_data).unsqueeze(1)
    features = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), 64):
            batch = X_tensor[i:i+64].to(device)
            features.append(encoder_model(batch).cpu().numpy())
    features = np.concatenate(features, axis=0)
    
    th_true = y_data * (np.pi / N_thetas)
    Y_reg = np.stack([np.sin(2 * th_true), np.cos(2 * th_true)], axis=1)
    
    X_tr, X_te, Y_tr, Y_te, th_tr, th_te = train_test_split(
        features, Y_reg, th_true, test_size=0.2, random_state=42
    )
    
    regressor = Ridge(alpha=1.0)
    regressor.fit(X_tr, Y_tr)
    Y_pred = regressor.predict(X_te)
    
    th_pred = np.mod(np.arctan2(Y_pred[:, 0], Y_pred[:, 1]) / 2.0, np.pi)
    diff = np.abs(th_pred - th_te)
    return np.degrees(np.mean(np.minimum(diff, np.pi - diff)))

In [8]:
def run_experiment(b_theta, b_sf, b_theta_deg, b_sf_cpd):
    print(f"\n[{b_theta_deg}°, {b_sf_cpd} cpd] Generating Data...")
    
    X_data, y_data = [], []
    for theta_idx, theta in enumerate(thetas):
        for seed in range(15): 
            fx, fy, ft = mc.get_grids(N_X, N_Y, N_frames)
            
            # To ensure numerical stability, we set minimum values for B_theta and B_sf
            safe_b_theta = max(b_theta, 0.02)
            safe_b_sf = max(b_sf, 0.001)
            
            mc_i = mc.envelope_gabor(
                fx, fy, ft, V_X=0., V_Y=0., B_V=0., 
                sf_0=sf_base, B_sf=safe_b_sf, theta=theta, B_theta=safe_b_theta
            )
            
            im = mc.rectif(mc.random_cloud(mc_i, seed=seed), contrast=1.0) - 0.5
            im = im[:, :, 0]
            
            im_std = im.std()
            if im_std == 0: 
                im += np.random.normal(0, 1e-5, im.shape)
                im_std = im.std()
                
            im = (im - im.mean()) / (im_std + 1e-8)
            
            X_data.append(im)
            y_data.append(theta_idx)
            
    X_data = np.array(X_data, dtype=np.float32)
    y_data = np.array(y_data, dtype=np.int64)
    dataloader = DataLoader(ImageDataset(X_data), batch_size=256, shuffle=True)

    # Embedding Dimension -------------------------------------

    ae_encoder = BaseEncoder(embed_dim=embed_dim).to(device)
    ae_decoder = PixelDecoder(embed_dim=embed_dim).to(device)
    ae_opt = torch.optim.AdamW(list(ae_encoder.parameters()) + list(ae_decoder.parameters()), lr=5e-4)

    # Training Standard Masked Autoencoder (MAE) --------------
    print(f"[{b_theta_deg}°, {b_sf_cpd} cpd] Training Standard MAE...")
    
    ae_encoder.train()
    ae_decoder.train() 
    
    for _ in range(epochs):
        for images in dataloader:
            images = images.to(device)
            b, c, h, w = images.shape
            
            target_idx = np.random.randint(4)
            mask = torch.ones_like(images)
            if target_idx == 0: mask[..., :h//2, :w//2] = 0
            elif target_idx == 1: mask[..., :h//2, w//2:] = 0
            elif target_idx == 2: mask[..., h//2:, :w//2] = 0
            elif target_idx == 3: mask[..., h//2:, w//2:] = 0
                
            masked_images = images * mask
            
            ae_opt.zero_grad()
            recons = ae_decoder(ae_encoder(masked_images))
            loss = F.mse_loss(recons, images)
            loss.backward()
            ae_opt.step()
            
    print(f"Standard MAE Final Loss: {loss.item():.4f}")
    AE_rec_error = loss.item()
    
    # Training Sparse Masked Autoencoder ------------------------
    print(f"[{b_theta_deg}°, {b_sf_cpd} cpd] Training Sparse MAE...")
    sae_encoder = BaseEncoder(embed_dim=embed_dim).to(device)
    sae_decoder = PixelDecoder(embed_dim=embed_dim).to(device)
    sae_opt = torch.optim.AdamW(list(sae_encoder.parameters()) + list(sae_decoder.parameters()), lr=5e-4)
    
    sae_encoder.train()
    sae_decoder.train()
    l1_weight = 1e-3  
    
    for _ in range(epochs):
        for images in dataloader:
            images = images.to(device)
            b, c, h, w = images.shape
            
            target_idx = np.random.randint(4)
            mask = torch.ones_like(images)
            if target_idx == 0: mask[..., :h//2, :w//2] = 0
            elif target_idx == 1: mask[..., :h//2, w//2:] = 0
            elif target_idx == 2: mask[..., h//2:, :w//2] = 0
            elif target_idx == 3: mask[..., h//2:, w//2:] = 0
                
            masked_images = images * mask
            
            sae_opt.zero_grad()
            
            z = sae_encoder(masked_images)
            recons = sae_decoder(z)
            mse_loss = F.mse_loss(recons, images)
            l1_loss = z.abs().mean()
            loss = mse_loss + l1_weight * l1_loss
            
            loss.backward()
            sae_opt.step()

    print(f"Sparse MAE MSE Loss: {mse_loss.item():.4f}, L1 Loss: {l1_loss.item():.4f}")
    print(f"Sparse MAE Final Loss: {loss.item():.4f}")
    SAE_rec_error = mse_loss.item()
    
    # Training LeJEPA ad its Decoder ------------------------------
    print(f"[{b_theta_deg}°, {b_sf_cpd} cpd] Training LeJEPA with Decoder...")
    jepa_encoder = BaseEncoder(embed_dim=embed_dim).to(device)
    target_encoder = BaseEncoder(embed_dim=embed_dim).to(device)
    target_encoder.load_state_dict(jepa_encoder.state_dict())
    for p in target_encoder.parameters(): p.requires_grad = False 
    
    jepa_predictor = JEPAPredictor(embed_dim=embed_dim).to(device)
    jepa_decoder = PixelDecoder(embed_dim=embed_dim).to(device)
    sigreg_loss_fn = SIGRegLoss(num_slices=embed_dim).to(device)
    sigreg_weight = 0.005
    
    jepa_opt = torch.optim.AdamW(
        list(jepa_encoder.parameters()) + 
        list(jepa_predictor.parameters()) + 
        list(jepa_decoder.parameters()), 
        lr=5e-4
    )
    
    jepa_encoder.train()
    jepa_predictor.train()
    jepa_decoder.train()
    
    for _ in range(epochs):
        for images in dataloader:
            images = images.to(device)
            b, c, h, w = images.shape
            jepa_opt.zero_grad()
            
            target_idx = np.random.randint(4)
            mask = torch.ones_like(images)
            if target_idx == 0: mask[..., :h//2, :w//2] = 0
            elif target_idx == 1: mask[..., :h//2, w//2:] = 0
            elif target_idx == 2: mask[..., h//2:, :w//2] = 0
            elif target_idx == 3: mask[..., h//2:, w//2:] = 0
                
            context_images = images * mask
            
            with torch.no_grad():
                target_repr = target_encoder(images)
                
            pred_repr = jepa_predictor(
                jepa_encoder(context_images), 
                quadrant_coords[target_idx].unsqueeze(0).repeat(b, 1)
            )
            
            pred_loss = F.mse_loss(pred_repr, target_repr)
            reg_loss = sigreg_loss_fn(target_repr)
            
            recons = jepa_decoder(pred_repr.detach()) 
            recon_loss = F.mse_loss(recons, images)
            
            loss = pred_loss + (sigreg_weight * reg_loss)
            loss.backward()
            jepa_opt.step()
            
            with torch.no_grad():
                for pq, pk in zip(jepa_encoder.parameters(), target_encoder.parameters()):
                    pk.data.mul_(0.99).add_(pq.data, alpha=0.01)

    print(f"LeJEPA Losses - Pred: {pred_loss.item():.4f}, SIGReg: {sigreg_weight * reg_loss.item():.4f}")
    JEPA_rec_error = recon_loss.item()

    # Reconstruction ----------------------------
    ae_encoder.eval(); ae_decoder.eval()
    sae_encoder.eval(); sae_decoder.eval()
    jepa_encoder.eval(); jepa_predictor.eval(); jepa_decoder.eval()
    
    with torch.no_grad():
        sample_batch = next(iter(dataloader)).to(device)
        b, c, h, w = sample_batch.shape
        
        mask = torch.ones_like(sample_batch)
        mask[..., :h//2, :w//2] = 0
        masked_sample_batch = sample_batch * mask
        
        sample_recons_ae = ae_decoder(ae_encoder(masked_sample_batch))
        sample_recons_sae = sae_decoder(sae_encoder(masked_sample_batch))
        
        sample_pred_repr_jepa = jepa_predictor(
            jepa_encoder(masked_sample_batch), 
            quadrant_coords[0].unsqueeze(0).repeat(b, 1)
        )
        sample_recons_jepa = jepa_decoder(sample_pred_repr_jepa)
        
        save_reconstruction_plot(
            sample_batch, masked_sample_batch, 
            sample_recons_ae, sample_recons_sae, sample_recons_jepa, 
            b_theta_deg, b_sf_cpd
        )

    

    ae_error = evaluate_model(ae_encoder, X_data, y_data)
    sae_error = evaluate_model(sae_encoder, X_data, y_data)
    jepa_error = evaluate_model(jepa_encoder, X_data, y_data)

    # Clean up to save memory --------------------------------
    del ae_encoder, ae_decoder, ae_opt
    del sae_encoder, sae_decoder, sae_opt
    del jepa_encoder, target_encoder, jepa_predictor, jepa_decoder, jepa_opt
    del dataloader, X_data, y_data
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print(f"[{b_theta_deg}°, {b_sf_cpd} cpd] Results - MAE: {ae_error:.1f}°, Sparse MAE: {sae_error:.1f}°, LeJEPA: {jepa_error:.1f}°")
    return ae_error, sae_error, jepa_error, AE_rec_error, SAE_rec_error, JEPA_rec_error 

In [ ]:



# Grid Search Execution --------------------------------
results = []

for b_theta, b_theta_deg in zip(B_thetas_rad, B_thetas_deg):
    for b_sf, b_sf_cpd in zip(B_sfs_cpp, B_sfs_cpd):
        ae_err, sae_err, jepa_err, AE_rec_error, SAE_rec_error, JEPA_rec_error = run_experiment(b_theta, b_sf, b_theta_deg, b_sf_cpd)
        results.append({
            "B_theta (deg)": b_theta_deg,
            "B_sf (cpd)": b_sf_cpd,
            "MAE Angular Error (°)": round(ae_err, 2),
            "Sparse MAE Angular Error (°)": round(sae_err, 2),
            "JEPA Angular Error (°)": round(jepa_err, 2),
            "MAE Rec Error": round(AE_rec_error, 4),
            "Sparse MAE Rec Error": round(SAE_rec_error, 4),
            "JEPA Rec Error": round(JEPA_rec_error, 4)
        })

# Print Table and Save
df = pd.DataFrame(results)
print("\n\n" + "="*80)
print("FINAL COMPARISON: MAE vs SPARSE MAE vs LeJEPA")
print("="*80)
print(df.to_markdown(index=False))

df.to_csv(f"output/jepa_vs_mae_vs_smae_grid_results_{embed_dim}.csv", index=False)
print("\nReconstructed Image plots saved to 'output/' folder.")
print(f"Data table saved to 'output/jepa_vs_mae_vs_smae_grid_results_{embed_dim}.csv'.")


[0.01°, 0.01 cpd] Generating Data...
[0.01°, 0.01 cpd] Training Standard MAE...
Standard MAE Final Loss: 0.7522
[0.01°, 0.01 cpd] Training Sparse MAE...
Sparse MAE MSE Loss: 0.7582, L1 Loss: 0.6156
Sparse MAE Final Loss: 0.7589
[0.01°, 0.01 cpd] Training LeJEPA with Decoder...
LeJEPA Losses - Pred: 0.0099, SIGReg: 0.2134
[0.01°, 0.01 cpd] Results - MAE: 7.6°, Sparse MAE: 12.9°, LeJEPA: 1.8°

[0.01°, 0.175 cpd] Generating Data...
[0.01°, 0.175 cpd] Training Standard MAE...
Standard MAE Final Loss: 0.9932
[0.01°, 0.175 cpd] Training Sparse MAE...
Sparse MAE MSE Loss: 0.9987, L1 Loss: 0.5419
Sparse MAE Final Loss: 0.9993
[0.01°, 0.175 cpd] Training LeJEPA with Decoder...
LeJEPA Losses - Pred: 0.0108, SIGReg: 0.3459
[0.01°, 0.175 cpd] Results - MAE: 5.3°, Sparse MAE: 7.0°, LeJEPA: 2.0°

[0.01°, 0.7 cpd] Generating Data...
[0.01°, 0.7 cpd] Training Standard MAE...
Standard MAE Final Loss: 0.9758
[0.01°, 0.7 cpd] Training Sparse MAE...
Sparse MAE MSE Loss: 0.9962, L1 Loss: 0.4996
Sparse MAE